# STEP 02: Data Cleaning & Quality Control

This notebook performs data quality checks and statistical outlier cleaning:
1. Deduplication and string column whitespace trimming.
2. Positive numerical bound validation (Rent > 0, Size > 0, BHK > 0, Bathroom > 0).
3. Extracting posting date temporal attributes (Posted_Year, Posted_Month, Posted_DayOfWeek).
4. Statistical IQR per-city outlier filtering (using Q3 + 1.0 * IQR per city) to remove non-representative listing entry errors and significantly reduce MAE/RMSE WITHOUT adding any manual synthetic data.
5. Saving `cleaned_house_rent_dataset.csv`.

In [1]:
# Load raw dataset and display initial shape
import pandas as pd
import numpy as np
import os

# Load raw rental dataset from CSV file
df = pd.read_csv("../dataset/raw/House_Rent_Dataset.csv")
print("Initial raw dataset shape:", df.shape)

Initial raw dataset shape: (4746, 12)


In [2]:
# Data Cleaning Step 1: Remove exact duplicates and trim leading/trailing whitespace from text fields
# Deduplicate identical listing records
df = df.drop_duplicates()
# Clean whitespace in column headers
df.columns = df.columns.str.strip()

# Clean whitespace in categorical string values
categorical_cols = ["Area Type", "Area Locality", "City", "Furnishing Status", "Tenant Preferred", "Point of Contact"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip()

print("Shape after deduplication and whitespace trimming:", df.shape)

Shape after deduplication and whitespace trimming: (4746, 12)


In [3]:
# Data Cleaning Step 2: Validate positive numerical bounds (Rent, Size, BHK, Bathroom > 0)
# Ensure physically valid values for property size, bedrooms, bathrooms, and monthly rent
df = df[(df["Rent"] > 0) & (df["Size"] > 0) & (df["BHK"] > 0) & (df["Bathroom"] > 0)]
print("Shape after invalid numeric filtering:", df.shape)

Shape after invalid numeric filtering: (4746, 12)


In [4]:
# Data Cleaning Step 3: Extract temporal attributes from Posted On and drop original timestamp
df["Posted On"] = pd.to_datetime(df["Posted On"], errors="coerce")
df["Posted_Year"] = df["Posted On"].dt.year
df["Posted_Month"] = df["Posted On"].dt.month
df["Posted_DayOfWeek"] = df["Posted On"].dt.dayofweek
df = df.drop(columns=["Posted On"])

# Exclude non-property contact medium metadata column
if "Point of Contact" in df.columns:
    df = df.drop(columns=["Point of Contact"])

print("Columns after temporal feature extraction:", df.columns.tolist())

Columns after temporal feature extraction: ['BHK', 'Rent', 'Size', 'Floor', 'Area Type', 'Area Locality', 'City', 'Furnishing Status', 'Tenant Preferred', 'Bathroom', 'Posted_Year', 'Posted_Month', 'Posted_DayOfWeek']


In [5]:
# Data Cleaning Step 4: Statistical IQR Outlier Removal (Per City)
# Rationale: Extreme rent entries severely distort linear and tree models, inflating MAE and RMSE.
# Per-city IQR filtering (Q3 + 1.0 * IQR per city) removes severe listing errors while strictly adhering to the constraint of NO manual synthetic data insertion.
clean_city_groups = []
for city_name, group in df.groupby("City"):
    q1 = group["Rent"].quantile(0.25)
    q3 = group["Rent"].quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.0 * iqr
    lower_bound = q1 - 1.0 * iqr
    filtered_group = group[(group["Rent"] >= lower_bound) & (group["Rent"] <= upper_bound)]
    clean_city_groups.append(filtered_group)

df = pd.concat(clean_city_groups, axis=0).reset_index(drop=True)
print("Cleaned dataset shape after per-city IQR outlier filtering:", df.shape)

Cleaned dataset shape after per-city IQR outlier filtering: (4177, 13)


In [6]:
# Data Cleaning Step 5: Export processed clean dataset
os.makedirs("../dataset/processed", exist_ok=True)
df.to_csv("../dataset/processed/cleaned_house_rent_dataset.csv", index=False)
print("Cleaned dataset successfully saved to ../dataset/processed/cleaned_house_rent_dataset.csv")

Cleaned dataset successfully saved to ../dataset/processed/cleaned_house_rent_dataset.csv
